# Sesión 07 - Modelos de probabilidad discretos

Objetivo: ajustar e interpretar modelos Bernoulli, Binomial, Geométrico, Poisson, Binomial Negativo y Multinomial.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(41)
pd.set_option("display.precision", 4)


## 1. Bernoulli y Binomial


In [ ]:
p_conversion = 0.08
n_visitas = 200

conversiones = stats.binom(n=n_visitas, p=p_conversion)
prob_al_menos_20 = 1 - conversiones.cdf(19)

print(f"E[X] = {conversiones.mean():.2f}")
print(f"Var(X) = {conversiones.var():.2f}")
print(f"P(X >= 20 conversiones) = {prob_al_menos_20:.4f}")


In [ ]:
ks = np.arange(0, 40)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ks, conversiones.pmf(ks))
ax.set_title("Binomial: conversiones en 200 visitas")
ax.set_xlabel("conversiones")
ax.set_ylabel("probabilidad")
plt.show()


## 2. Geométrica: intentos hasta primer éxito


In [ ]:
p_contacto = 0.25
geom = stats.geom(p=p_contacto)

print(f"Intentos esperados hasta contacto: {geom.mean():.2f}")
print(f"P(contactar en máximo 3 intentos) = {geom.cdf(3):.4f}")


## 3. Poisson para conteos


### Lectura matemática

- **Distribución asumida:** $X\sim Poisson(\lambda)$ para conteos en un intervalo.
- **Parámetro estimado/usado:** tasa $\lambda=E[X]=Var(X)$.
- **Supuesto que puede fallar:** sobredispersión, exceso de ceros o dependencia entre eventos.
- **Diagnóstico:** comparar media vs varianza y frecuencias observadas vs esperadas.


In [ ]:
lam = 4.2
reclamos = stats.poisson(mu=lam)
ks = np.arange(0, 15)

print(f"P(0 reclamos) = {reclamos.pmf(0):.4f}")
print(f"P(5 o más reclamos) = {1 - reclamos.cdf(4):.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ks, reclamos.pmf(ks))
ax.set_title("Poisson: reclamos por hora")
ax.set_xlabel("reclamos")
ax.set_ylabel("probabilidad")
plt.show()


## 4. Sobredispersión: Poisson vs Binomial Negativa

Si la varianza observada supera mucho a la media, Poisson suele subestimar colas.


In [ ]:
conteos = rng.negative_binomial(n=2.5, p=0.38, size=2_000)
media = conteos.mean()
varianza = conteos.var(ddof=0)

r_mom = media ** 2 / (varianza - media)
p_mom = r_mom / (r_mom + media)

print(f"media empírica = {media:.3f}")
print(f"varianza empírica = {varianza:.3f}")
print(f"r estimado NB = {r_mom:.3f}")
print(f"p estimado NB = {p_mom:.3f}")


In [ ]:
ks = np.arange(0, np.percentile(conteos, 99).astype(int) + 1)
frec = pd.Series(conteos).value_counts(normalize=True).reindex(ks, fill_value=0)
pois_fit = stats.poisson(mu=media)
nb_fit = stats.nbinom(n=r_mom, p=p_mom)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ks, frec.values, alpha=0.45, label="datos")
ax.plot(ks, pois_fit.pmf(ks), marker="o", label="Poisson ajustada")
ax.plot(ks, nb_fit.pmf(ks), marker="o", label="NegBin ajustada")
ax.set_title("Comparación para conteos sobredispersos")
ax.set_xlabel("conteo")
ax.set_ylabel("probabilidad")
ax.legend()
plt.show()


## 5. Multinomial


In [ ]:
categorias = ["web", "tienda", "call_center", "app"]
probs = np.array([0.35, 0.25, 0.10, 0.30])
n_clientes = 100

muestra = stats.multinomial(n=n_clientes, p=probs).rvs(random_state=rng)[0]
pd.DataFrame({"canal": categorias, "probabilidad": probs, "clientes_simulados": muestra})


## 6. Modelos discretos con churn y resultados multiclase

Este bloque reutiliza `telecom_churn.csv` para Bernoulli/conteos y `results_multinomial.csv` para discutir clasificación multiclase del material anterior.


### Lectura matemática

- **Distribución asumida:** Bernoulli para churn y Categórica/Multinomial para clases.
- **Parámetros estimados:** $P(Y=1\mid X)$ y probabilidades por clase.
- **Supuesto que puede fallar:** probabilidades mal calibradas o clases desbalanceadas.
- **Diagnóstico:** log-loss, Brier, PR-AUC y calibración.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "telecom_churn.csv").exists():
    print("No se encontró data_sources/telecom_churn.csv. Se mantiene la sección sintética.")
else:
    churn_df = pd.read_csv(DATA_DIR / "telecom_churn.csv")
    p_churn = churn_df["Churn"].mean()
    llamadas = churn_df["CustServCalls"]
    print(f"P(Churn=1) Bernoulli estimada = {p_churn:.4f}")
    print(f"Media CustServCalls = {llamadas.mean():.4f}")
    print(f"Varianza CustServCalls = {llamadas.var(ddof=0):.4f}")
    print("Varianza > media sugiere sobredispersión frente a Poisson." if llamadas.var(ddof=0) > llamadas.mean() else "Poisson no queda descartada por media-varianza.")

    kmax = int(llamadas.quantile(0.99))
    ks = np.arange(0, kmax + 1)
    frec = llamadas.value_counts(normalize=True).reindex(ks, fill_value=0)
    pois = stats.poisson(mu=llamadas.mean())
    display(pd.DataFrame({"k": ks, "obs": frec.values, "poisson": pois.pmf(ks)}).head(10))


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "results_multinomial.csv").exists():
    resultados_multi = pd.read_csv(DATA_DIR / "results_multinomial.csv")
    display(resultados_multi.sort_values("logloss"))


## 7. Funciones de costo: log-loss, pesos de clase y focal loss

Este bloque recupera la exploración de pérdidas del legacy. Para Bernoulli, la log-loss es la verosimilitud negativa. Focal loss modifica esa pérdida para reducir el peso de ejemplos fáciles:

$$
FL(p_t)=-\alpha_t(1-p_t)^\gamma\log(p_t)
$$

Con $\gamma>0$, si un ejemplo ya tiene $p_t$ alto, contribuye menos al gradiente.


### Lectura matemática

- **Objetivo asumido:** minimizar una pérdida compatible con la decisión.
- **Parámetros estimados:** probabilidades y umbral de decisión.
- **Supuesto que puede fallar:** optimizar una métrica que no representa el costo real.
- **Diagnóstico:** matriz de confusión por umbral, costo esperado y calibración.


In [ ]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

if DATA_DIR is None or not (DATA_DIR / "telecom_churn.csv").exists():
    print("No se encontró telecom_churn.csv. Se omite el bloque de pérdidas supervisadas.")
else:
    churn_df = pd.read_csv(DATA_DIR / "telecom_churn.csv")
    X = churn_df.drop(columns="Churn")
    y = churn_df["Churn"].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

    modelos = {
        "baseline_prior": DummyClassifier(strategy="prior"),
        "log_reg_logloss": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
        "log_reg_balanced": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    }

    filas = []
    probas = {}
    for nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        p_hat = modelo.predict_proba(X_test)[:, 1]
        probas[nombre] = p_hat
        filas.append(
            {
                "modelo": nombre,
                "log_loss": log_loss(y_test, p_hat),
                "brier": brier_score_loss(y_test, p_hat),
                "roc_auc": roc_auc_score(y_test, p_hat),
                "pr_auc": average_precision_score(y_test, p_hat),
            }
        )

    def focal_loss(y_true, p_hat, alpha=0.25, gamma=2.0, eps=1e-12):
        y_true = np.asarray(y_true)
        p_hat = np.clip(np.asarray(p_hat), eps, 1 - eps)
        p_t = np.where(y_true == 1, p_hat, 1 - p_hat)
        alpha_t = np.where(y_true == 1, alpha, 1 - alpha)
        return np.mean(-alpha_t * (1 - p_t) ** gamma * np.log(p_t))

    for row in filas:
        row["focal_loss_eval"] = focal_loss(y_test, probas[row["modelo"]], alpha=0.75, gamma=2.0)

    display(pd.DataFrame(filas).sort_values("log_loss"))


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "telecom_churn.csv").exists():
    thresholds = np.linspace(0.05, 0.95, 91)
    costo_fp = 1.0
    costo_fn = 6.0
    p_ref = probas["log_reg_balanced"]
    costos = []
    for thr in thresholds:
        pred = (p_ref >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        costo = costo_fp * fp + costo_fn * fn
        costos.append({"threshold": thr, "costo_total": costo, "fp": fp, "fn": fn, "tp": tp, "tn": tn})
    costos_df = pd.DataFrame(costos).sort_values("costo_total")
    display(costos_df.head(10))


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "telecom_churn.csv").exists():
    try:
        from catboost import CatBoostClassifier
        cb_modelos = {
            "catboost_logloss": CatBoostClassifier(loss_function="Logloss", iterations=80, depth=4, learning_rate=0.08, random_seed=7, verbose=False),
            "catboost_focal": CatBoostClassifier(loss_function="Focal:focal_alpha=0.75;focal_gamma=2", iterations=80, depth=4, learning_rate=0.08, random_seed=7, verbose=False),
        }
        cb_rows = []
        for nombre, modelo in cb_modelos.items():
            modelo.fit(X_train, y_train)
            p_hat = modelo.predict_proba(X_test)[:, 1]
            cb_rows.append(
                {
                    "modelo": nombre,
                    "log_loss": log_loss(y_test, p_hat),
                    "brier": brier_score_loss(y_test, p_hat),
                    "roc_auc": roc_auc_score(y_test, p_hat),
                    "pr_auc": average_precision_score(y_test, p_hat),
                    "focal_loss_eval": focal_loss(y_test, p_hat, alpha=0.75, gamma=2.0),
                }
            )
        display(pd.DataFrame(cb_rows).sort_values("focal_loss_eval"))
    except Exception as exc:
        print("CatBoost/Focal no disponible en este entorno; se conserva la comparación con sklearn.")
        print(type(exc).__name__, exc)


## Práctica

Simula otro conteo con sobredispersión y compara media, varianza, Poisson y Binomial Negativa. Explica cuál modelo usarías y por qué.
